<a href="https://colab.research.google.com/github/Jayanth2006-24/1BM24CS124-AI/blob/main/1BM24CS124_Week7_Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re

def parse(expr):
    tokens = re.findall(r'[a-zA-Z_]\w*|[(),]', expr)
    pos = 0

    def build():
        nonlocal pos

        name = tokens[pos]
        pos += 1

        if pos < len(tokens) and tokens[pos] == '(':
            pos += 1
            args = []

            while True:
                args.append(build())

                if tokens[pos] == ')':
                    pos += 1
                    break

                pos += 1

            return (name, args)

        return name

    return build()


def is_variable(term):
    return isinstance(term, str) and (
        term in {'x', 'y', 'z'} or term.startswith('?')
    )


def apply_substitution(term, subst):
    if isinstance(term, str):
        while is_variable(term) and term in subst:
            term = subst[term]
        return term

    if isinstance(term, tuple):
        return (
            term[0],
            [apply_substitution(arg, subst) for arg in term[1]]
        )

    return term


def occurs(var, term, subst):
    term = apply_substitution(term, subst)

    if var == term:
        return True

    if isinstance(term, tuple):
        return any(occurs(var, arg, subst) for arg in term[1])

    return False


def unify(t1, t2, subst):
    t1 = apply_substitution(t1, subst)
    t2 = apply_substitution(t2, subst)

    if t1 == t2:
        return True

    if is_variable(t1):
        if occurs(t1, t2, subst):
            return False
        subst[t1] = t2
        return True

    if is_variable(t2):
        if occurs(t2, t1, subst):
            return False
        subst[t2] = t1
        return True

    if isinstance(t1, tuple) and isinstance(t2, tuple):

        if t1[0] != t2[0]:
            return False

        if len(t1[1]) != len(t2[1]):
            return False

        for a, b in zip(t1[1], t2[1]):
            if not unify(a, b, subst):
                return False

        return True

    return False


def display(term):
    if isinstance(term, tuple):
        return term[0] + "(" + ", ".join(
            display(arg) for arg in term[1]
        ) + ")"
    return term


print("FIRST ORDER LOGIC - UNIFICATION")

expr1 = input("Enter first expression: ")
expr2 = input("Enter second expression: ")

try:
    t1 = parse(expr1)
    t2 = parse(expr2)

    if isinstance(t1, tuple) and isinstance(t2, tuple):
        if t1[0] != t2[0]:
            print("Unification Failed!")
        elif len(t1[1]) != len(t2[1]):
            print("Unification Failed!")
        else:
            subst = {}

            if unify(t1, t2, subst):
                print("\nUnification Successful!")
                print("Most General Unifier (MGU):")

                if subst:
                    for var, value in subst.items():
                        print(var, "=", display(value))
                else:
                    print("No substitution required (NIL)")
            else:
                print("\nUnification Failed!")
                print("Reason: Conflicting terms or occurs check failed.")

    elif unify(t1, t2, {}):
        print("\nUnification Successful!")
    else:
        print("\nUnification Failed!")

except (IndexError, ValueError):
    print("Invalid expression!")


FIRST ORDER LOGIC - UNIFICATION
Enter first expression: p(b,x,f(g(z)))
Enter second expression: : p(z,f(y),f(y))

Unification Successful!
Most General Unifier (MGU):
z = b
x = f(y)
y = g(b)
